# 在 Colab 中打开
<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ai-agents-the-definitive-guide/blob/main/CH01/ch01_code_examples.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Notebook 说明

本 Notebook 是一个紧凑的端到端入门示例：从一次普通的大语言模型（LLM）调用出发，逐步构建一个能够调用工具、跨轮次保存状态，并追踪自身执行流程的小型但真实的 Agent。

## 这里会展示什么

* 使用 `langchain-openai` 完成一次**无状态 LLM 调用**，作为热身。
* 使用两个简单工具构建**循环式工具调用**：

  1. 通过 SerpAPI 实现 `internet_search`，获取最新信息；
  2. 通过 `numexpr` 实现 `calculator`，进行快速数学计算。
* 构建一个**最小化 LangGraph Agent**，其中包括：

  * 一个带类型的状态（typed state），使用 `add_messages` 累积 `messages`；
  * 一个绑定工具的 `llm` 节点，以及负责执行工具的 `ToolNode`；
  * 一个路由器（router），决定何时调用工具、何时结束；
  * 使用 `MemorySaver` 实现内存中的检查点（checkpoint）；
  * 使用不同线程（thread）分叉对话，并比较各自的记忆。

## 为什么选择这些示例

1. 先看到一个**最小工具调用循环**，它不依赖任何框架级状态。
2. 再看到如何用 LangGraph 把同一个思路工程化：获得更清晰的路由、检查点和线程级记忆。
3. 最后加入**追踪工具（trace utilities）**，打印节点更新和简短状态快照，便于调试和教学。

## 如何运行

* 从 `.env` 加载密钥，或用 `%env` 设置。若缺失，Notebook 会交互式询问。
* 先调用 `gpt-5-mini`（也可以换成任何兼容 OpenAI API 的 LLM）做快速检查，再切换到绑定工具的 `gpt-4o`。
* 运行一个**两步任务**：
  * 用 `internet_search` 获取**纽约市当前气温**；
  * 用 `calculator` 计算该温度的**平方**。
* 然后在 **LangGraph** 应用中重复任务，再分叉出一个**第二线程**，把温度转换为华氏度并同时报告摄氏和华氏结果。
  你会看到逐节点更新，以及每个线程最终的记忆快照。

## 需要重点观察的核心思想

* **把工具绑定到模型**，让模型自行决定什么时候调用。
* 一个**最小路由器**：检查是否存在 `tool_calls`，若有则进入 `ToolNode`，否则结束。
* 使用 **thread id** 隔离不同记忆，并支持可复现的调试。
* 在需要时采用**确定性设置**：`temperature=0`、显式 `recursion_limit` 和简短格式提示，以提高输出一致性。

## 如何替换和扩展

* 可以把 `SerpAPIWrapper` 换成任何能返回文本的搜索工具。
* 可以新增自己的工具，并加入 `tools` 和 `tool_map`。
* 如果需要长期会话，可以把 `MemorySaver` 换成持久化 checkpointer。
* 如果需要更大的图，可以增加规划、校验或护栏（guardrails）节点。

## 依赖与注意事项

* 需要可用的 [`OPENAI_API_KEY`](https://platform.openai.com/api-keys) 和 [`SerpAPIWrapper`](https://serpapi.com/)。
* 互联网搜索结果会变化，因此示例中的气温和搜索摘要会随时间和来源变化。
* 工具调用会消耗 token，也可能产生外部 API 费用，需要关注成本。


# 依赖安装

In [ ]:
!pip install -q langgraph==0.6.7 langchain-openai==0.3.33 python-dotenv==1.1.1 langchain_community google-search-results

  Preparing metadata (setup.py) ... done


# API 相关导入

In [ ]:

from dotenv import load_dotenv
import os

# 导入依赖

In [ ]:
# 标准库
import os
import math
import json
import numexpr
from typing import List, Dict, Any, TypedDict, Annotated

# LangChain 核心组件
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain_community.utilities import SerpAPIWrapper

# LangGraph
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver


In [ ]:
# --- API 密钥设置 ---
# 方式 1（推荐）：在项目目录创建 `.env` 文件，并写入：
# OPENAI_API_KEY=your_openai_key_here
# SERPAPI_API_KEY=your_serpapi_key_here
#
# 方式 2：直接在 Notebook 中使用 magic 命令设置：
# %env OPENAI_API_KEY=your_openai_key_here
# %env SERPAPI_API_KEY=your_serpapi_key_here

from dotenv import load_dotenv
import os

# 如果存在 .env，则从中加载
load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
serp_api_key = os.getenv("SERPAPI_API_KEY")

# 兜底：如果仍缺失，则交互式询问
if not OPENAI_API_KEY:
    print("⚠️ OPENAI_API_KEY not found. You can set it with `%env` in the notebook or enter it below.")
    OPENAI_API_KEY = input("Enter your OPENAI_API_KEY: ").strip()

if not serp_api_key:
    print("⚠️ SERPAPI_API_KEY not found. You can set it with `%env` in the notebook or enter it below.")
    serp_api_key = input("Enter your SERPAPI_API_KEY: ").strip()

print("✅ API keys loaded successfully!")



# 第一个示例：使用 LangChain 进行无状态 LLM 调用
____

## 配置 LLM

In [ ]:
llm = ChatOpenAI(model="gpt-5-mini")
response = llm.invoke("What are AI agents?")
print(response.content)


Short answer
An AI agent is a software (or embodied) system that perceives its environment, makes decisions, and takes actions to achieve goals — usually with some degree of autonomy and adaptability.

Key characteristics
- Perception: senses inputs (camera, mic, sensors, API data, user text).
- Decision-making: chooses actions based on goals, models, or learned policies.
- Action: affects the environment (move a robot, send a message, place a trade).
- Autonomy: operates without requiring step-by-step human control.
- Goal-directedness: usually tries to maximize a reward or satisfy objectives.

Common types and examples
- Reactive agents: map inputs directly to actions (simple controllers, thermostats).
- Deliberative/planning agents: build internal models and plan ahead (robot path planners).
- Learning agents: improve with experience (reinforcement learning agents, recommendation systems).
- Hybrid agents: combine rules, planning, and learning.
- Embodied agents: robots, autonomous 

## 为无状态运行定义工具


## 工具

In [ ]:
@tool("internet_search")
def internet_search(query: str) -> str:
    """通过 SerpAPI 搜索 Google，获取最新信息。"""
    serp_api_key = os.environ["SERPAPI_API_KEY"]
    params = {"engine": "google", "gl": "us", "hl": "en"}
    search = SerpAPIWrapper(params=params, serpapi_api_key=serp_api_key)
    return search.run(query)

@tool("calculator")
def calculator(expression: str) -> str:
    """使用 numexpr 计算单行数学表达式。"""
    local_dict = {"pi": math.pi, "e": math.e}
    out = numexpr.evaluate(
        expression.strip(),
        global_dict={},
        local_dict=local_dict,
    )
    return str(out)

tools = [internet_search, calculator]

tool_map: Dict[str, Any] = {t.name: t for t in tools}

## 将模型绑定到工具

In [ ]:
llm = ChatOpenAI(model="gpt-4o").bind_tools(tools, tool_choice="any")

## 带工具循环的一次性无状态运行

In [ ]:
def run_once(prompt: str, max_steps: int = 4) -> str:
    messages = [HumanMessage(content=prompt)]
    for _ in range(max_steps):
        ai: AIMessage = llm.invoke(messages)
        messages.append(ai)

        calls = getattr(ai, "tool_calls", None) or []
        if not calls:
            break

        for call in calls:
            name = call["name"]
            args = call.get("args", {})
            result = tool_map[name].invoke(args)
            messages.append(ToolMessage(
                content=str(result),
                name=name,
                tool_call_id=call["id"]
            ))
    return messages[-1].content

print(run_once("""Two step task.

Step 1: Use internet_search to get the current air temperature in New York City today. Show the exact query you used, the top source title and snippet, and extract a numeric temperature in Celsius. Return this temperature as feedback for Step 2.

Step 2: Using the Celsius value from Step 1, compute its square with calculator. Show the exact expression you used and the numeric result.

Important: Give a short final answer in this format:
Current temperature:
Square of current temperature:"""))



256


# 工具

In [ ]:
@tool("internet_search")
def internet_search(query: str) -> str:
    """通过 SerpAPI 搜索 Google，获取最新信息。"""
    serp_api_key = os.environ["SERPAPI_API_KEY"]
    params = {"engine": "google", "gl": "us", "hl": "en"}
    search = SerpAPIWrapper(params=params, serpapi_api_key=serp_api_key)
    return search.run(query)

@tool("calculator")
def calculator(expression: str) -> str:
    """使用 numexpr 计算单行数学表达式。"""
    local_dict = {"pi": math.pi, "e": math.e}
    out = numexpr.evaluate(
        expression.strip(),
        global_dict={},
        local_dict=local_dict,
    )
    return str(out)

tools = [internet_search, calculator]
tool_map: Dict[str, Any] = {t.name: t for t in tools}


# 将模型绑定到工具

In [ ]:
llm = ChatOpenAI(model="gpt-4o", temperature=0, max_tokens=800).bind_tools(tools, tool_choice="auto")

# 最小工具循环（无状态）

In [ ]:

def run_once(prompt: str, max_steps: int = 8) -> str:
    messages: List[HumanMessage | AIMessage | ToolMessage] = [HumanMessage(content=prompt)]
    last_ai: AIMessage | None = None

    for _ in range(max_steps):
        ai: AIMessage = llm.invoke(messages)
        messages.append(ai)
        last_ai = ai

        calls = getattr(ai, "tool_calls", None) or []
        if not calls:
            # 模型已经给出最终答案
            return messages[-1].content

        # 执行工具调用，并把观察结果反馈给模型
        for call in calls:
            name = call["name"]
            args = call.get("args", {}) or {}
            result = tool_map[name].invoke(args)
            messages.append(ToolMessage(
                content=str(result),
                name=name,
                tool_call_id=call.get("id")
            ))

    # 如果循环结束时仍没有干净的最终 AI 消息，则强制收尾
    messages.append(HumanMessage(content="""
Finish now. Give a short final answer in this exact format:

Current temperature:
Square of current temperature:
""".strip()))
    final_ai: AIMessage = llm.invoke(messages)
    return final_ai.content

print(run_once("""Two step task.

Step 1: Use internet_search to get the current air temperature in New York City today. Show the exact query you used, the top source title and snippet, and extract a numeric temperature in Celsius. Return this temperature as feedback for Step 2.

Step 2: Using the Celsius value from Step 1, compute its square with calculator. Show the exact expression you used and the numeric result.

Important: Give a short final answer in this format:
Current temperature:
Square of current temperature:"""))


Current temperature: 16°C
Square of current temperature: 256


## 工具

In [ ]:
@tool("internet_search")
def internet_search(query: str) -> str:
    """通过 SerpAPI 搜索 Google，获取最新信息。"""
    serp_api_key = os.environ["SERPAPI_API_KEY"]
    params = {"engine": "google", "gl": "us", "hl": "en"}
    search = SerpAPIWrapper(params=params, serpapi_api_key=serp_api_key)
    return search.run(query)

@tool("calculator")
def calculator(expression: str) -> str:
    """使用 numexpr 计算单行数学表达式。"""
    local_dict = {"pi": math.pi, "e": math.e}
    out = numexpr.evaluate(
        expression.strip(),
        global_dict={},
        local_dict=local_dict,
    )
    return str(out)

tools = [internet_search, calculator]


## 构建一个包含状态、节点与路由的最小 LangGraph

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

# LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0, max_tokens=800).bind_tools(tools)

def llm_node(state: AgentState) -> AgentState:
    ai = llm.invoke(state["messages"])
    return {"messages": [ai]}

tool_node = ToolNode(tools=tools)

graph = StateGraph(AgentState)
graph.add_node("llm", llm_node)
graph.add_node("tools", tool_node)
graph.add_edge(START, "llm")

def route(state: AgentState):
    last = state["messages"][-1]
    calls = getattr(last, "tool_calls", None) or []
    return "tools" if calls else END

graph.add_conditional_edges("llm", route, {"tools": "tools", END: END})
graph.add_edge("tools", "llm")

## 使用内存检查点编译，并配置线程

In [ ]:
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)


## 追踪执行，并检查状态和记忆

In [ ]:
def _short(msg: BaseMessage, max_len: int = 140) -> str:
    """以紧凑单行形式展示一条消息。"""
    role = type(msg).__name__.replace("Message", "").lower()
    content = getattr(msg, "content", "")
    if isinstance(content, list):
        # 某些工具输出可能是列表形式的 payload
        try:
            content = json.dumps(content)
        except Exception:
            content = str(content)
    text = str(content).replace("\n", " ").strip()
    if len(text) > max_len:
        text = text[: max_len - 3] + "..."
    # 若存在，则附带工具名或函数调用信息
    if hasattr(msg, "tool_calls") and getattr(msg, "tool_calls"):
        tnames = [tc.get("name", "tool") for tc in msg.tool_calls]
        return f"{role}: tool_calls -> {tnames}"
    if isinstance(msg, ToolMessage):
        return f"{role}({msg.name}): {text}"
    return f"{role}: {text}"

## 追踪执行，并检查状态和记忆

In [ ]:
def print_state_snapshot(app, config, title: str):
    """打印指定线程当前的图状态和记忆。"""
    snap = app.get_state(config)
    values = snap.values or {}
    msgs: List[BaseMessage] = values.get("messages", [])
    print(f"\n=== {title} | state snapshot ===")
    print(f"messages: {len(msgs)} total")
    for i, m in enumerate(msgs[-5:], start=max(0, len(msgs)-5) + 1):
        print(f"  {i:>3}: {_short(m)}")
    # 如果存在，则展示路由信息和排队任务
    nxt = getattr(snap, "next", None)
    tasks = getattr(snap, "tasks", None)
    if nxt:
        print(f"next nodes: {list(nxt)}")
    if tasks:
        print(f"queued tasks: {tasks}")
    # 通过 checkpointer 展示该线程的最小记忆视图
    # 默认情况下，MemorySaver 会为每个线程保存最新 checkpoint，因此这里只确认其存在
    print("memory: in-memory checkpoint present for this thread")

## 追踪执行，并检查状态和记忆

In [ ]:
def run_with_tracing(app, input_state: AgentState, config, title: str):
    """运行图，同时打印逐节点更新和最终记忆。"""
    print(f"\n=== {title} | execution trace ===")
    final = None
    # stream_mode="updates" 会暴露节点级更新
    for event in app.stream(input_state, config=config, stream_mode="updates"):
        for node, upd in event.items():
            # upd 通常类似 {"messages": [<new msg>]}，也可能是工具结果
            keys = list(upd.keys())
            print(f"[enter {node}] updated: {keys}")
            # 如果 messages 有更新，则简要打印最后一条
            msgs = upd.get("messages") or []
            if msgs:
                print(f"  {_short(msgs[-1])}")
            print(f"[leave {node}]")
            final = upd
    # 从 app.get_state 中展示最终 assistant 消息
    print_state_snapshot(app, config, title=f"{title} | after run")
    snap = app.get_state(config)
    msgs = snap.values.get("messages", [])
    return msgs[-1].content if msgs else ""

In [ ]:
# 配置
cfg = {"configurable": {"thread_id": "nyc-weather-session"}}

## 第 1 轮：获取纽约市当前摄氏气温

In [ ]:
turn1_answer = run_with_tracing(
    app,
    {"messages": [HumanMessage(content="Get the current air temperature in New York City in Celsius.")]},
    config={**cfg, "recursion_limit": 20},
    title="TURN 1",
)
print("\nTURN 1 (final assistant):\n", turn1_answer)

## 第 2 轮：在同一线程中计算该温度的平方

In [ ]:
turn2_answer = run_with_tracing(
    app,
    {"messages": [HumanMessage(content="Now compute the square of that temperature.")]},
    config={**cfg, "recursion_limit": 20},
    title="TURN 2",
)
print("\nTURN 2 (final assistant):\n", turn2_answer)


## 分叉出一个并行线程，执行不同的后续任务

In [ ]:
cfg_branch = {"configurable": {"thread_id": "nyc-weather-session-branch"}}
branch_answer = run_with_tracing(
    app,
    {"messages": [HumanMessage(content="Instead of squaring, convert it to Fahrenheit and report both.")]},
    config=cfg_branch,
    title="BRANCH",
)
print("\nBRANCH (final assistant):\n", branch_answer)


In [ ]:



# 展示两个线程汇总后的记忆视图
print_state_snapshot(app, cfg, title="MAIN THREAD memory view")
print_state_snapshot(app, cfg_branch, title="BRANCH THREAD memory view")



=== TURN 1 | execution trace ===
[enter llm] updated: ['messages']
  ai: tool_calls -> ['internet_search']
[leave llm]
[enter tools] updated: ['messages']
  tool(internet_search): {'type': 'weather_result', 'temperature': '18', 'unit': 'Celsius', 'precipitation': '0%', 'humidity': '80%', 'wind': '23 km/h', 'location...
[leave tools]
[enter llm] updated: ['messages']
  ai: The current air temperature in New York City is 18°C.
[leave llm]

=== TURN 1 | after run | state snapshot ===
messages: 4 total
    1: human: Get the current air temperature in New York City in Celsius.
    2: ai: tool_calls -> ['internet_search']
    3: tool(internet_search): {'type': 'weather_result', 'temperature': '18', 'unit': 'Celsius', 'precipitation': '0%', 'humidity': '80%', 'wind': '23 km/h', 'location...
    4: ai: The current air temperature in New York City is 18°C.
memory: in-memory checkpoint present for this thread

TURN 1 (final assistant):
 The current air temperature in New York City is 18°C.

===